# 📅 Time Series Foundations
**fpppy Chapters 1–3 · Pattern Recognition for the Rest of Us**

> A time series is any sequence of observations indexed by time. Before forecasting, you need to understand the structure of your series — trend, seasonality, cycles, and noise. This notebook covers the essentials.

### Required reading
📘 [Forecasting: Principles & Practice (Pythonic) — Ch 1–3](https://otexts.com/fpppy/)

### What you'll learn
- Time series components: trend, seasonality, cycles, remainder
- Time plots, seasonal plots, autocorrelation (ACF/PACF)
- STL decomposition — separating components
- Stationarity — what it means and why it matters
- White noise — the baseline

### Dataset: Air passengers + Australian retail turnover
### Time: ~60 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import matplotlib.dates as mdates

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: AirPassengers

In [ ]:
import pandas as pd
passengers = pd.read_csv(f'{DATA_DIR}/AirPassengers.csv',
                          parse_dates=['Month'], index_col='Month')
passengers.index.freq = 'MS'
print(f"Air Passengers: {passengers.shape}  ({passengers.index[0].year}–{passengers.index[-1].year})")
passengers.head()

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

## 📊 Part 1 — Time Plots: See the Big Picture

Always start with a time plot. Look for:
- **Trend:** long-run increase or decrease
- **Seasonality:** regular, calendar-related patterns (monthly, quarterly, yearly)
- **Cycles:** irregular longer-term fluctuations (not calendar-based)
- **Outliers/irregularities:** sudden jumps, one-off events

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Raw series
axes[0].plot(passengers.index, passengers['Passengers'], color='#1e5fa8', lw=1.5)
axes[0].fill_between(passengers.index, passengers['Passengers'], alpha=0.1, color='#1e5fa8')
axes[0].set_title('International Air Passengers (1949–1960)')
axes[0].set_ylabel('Passengers (thousands)')

# Log transform to stabilize variance
log_pass = np.log(passengers['Passengers'])
axes[1].plot(passengers.index, log_pass, color='#e85d20', lw=1.5)
axes[1].set_title('Log(Passengers) — variance stabilized')
axes[1].set_ylabel('log(Passengers)')

plt.tight_layout()
plt.show()
print("📌 Clear upward trend + increasing seasonal swings → multiplicative seasonality")
print("   Log transform converts multiplicative → additive seasonality (constant swing size)")

In [ ]:
# Seasonal plot — one line per year
fig, ax = plt.subplots(figsize=(11, 5))
years = passengers.index.year.unique()
colors_y = plt.cm.viridis(np.linspace(0, 1, len(years)))
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for year, color in zip(years, colors_y):
    year_data = passengers[passengers.index.year == year]['Passengers'].values
    if len(year_data) == 12:
        ax.plot(range(12), year_data, color=color, lw=1.5, marker='o', markersize=4, label=str(year))

ax.set_xticks(range(12))
ax.set_xticklabels(months)
ax.set_ylabel('Passengers (thousands)')
ax.set_title('Seasonal Plot — Air Passengers by Month\n(one line per year, darker = later)')
ax.legend(title='Year', ncol=4, fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()
print("\n📌 July-August peak every year — summer travel season")
print("   Each year's line is higher → upward trend")
print("   The gap between lines grows → multiplicative seasonality")

## 🔀 Part 2 — Decomposition: Separating the Components

**Additive:** Y_t = Trend_t + Seasonal_t + Remainder_t (use when seasonal amplitude is constant)

**Multiplicative:** Y_t = Trend_t × Seasonal_t × Remainder_t (use when amplitude grows with trend)

**STL (Seasonal and Trend decomposition using Loess)** is more robust than classical decomposition — handles outliers better and allows the seasonal component to change over time.

In [ ]:
# Classical decomposition
decomp = seasonal_decompose(passengers['Passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
titles = ['Observed', 'Trend', 'Seasonal', 'Residual']
components = [passengers['Passengers'], decomp.trend, decomp.seasonal, decomp.resid]
colors_d = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1']

for ax, title, comp, color in zip(axes, titles, components, colors_d):
    ax.plot(passengers.index, comp, color=color, lw=1.5)
    ax.set_title(title)
    ax.set_ylabel(title)

plt.suptitle('Classical Multiplicative Decomposition — Air Passengers', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 Seasonal component: a consistent multiplier (1.3 in summer = 30% above trend)")
print("   Trend: smooth upward curve")
print("   Residual: what's left after removing trend and seasonality — should look random")

In [ ]:
# STL Decomposition — more robust
stl = STL(np.log(passengers['Passengers']), period=12, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 9)
fig.suptitle('STL Decomposition — log(Air Passengers)\nRobust to outliers, seasonal component can evolve', y=1.01)
plt.tight_layout()
plt.show()

# Strength measures
var_remainder = np.var(result.resid)
var_seasonal_remainder = np.var(result.seasonal + result.resid)
var_trend_remainder    = np.var(result.trend + result.resid)

F_seasonal = max(0, 1 - var_remainder/var_seasonal_remainder)
F_trend    = max(0, 1 - var_remainder/var_trend_remainder)
print(f"\nStrength of trend:     {F_trend:.3f}  (0=none, 1=strong)")
print(f"Strength of seasonality: {F_seasonal:.3f}  (0=none, 1=strong)")
print("\n📌 Both are near 1.0 → very strong trend and seasonality in this series")

## 📉 Part 3 — Autocorrelation: Memory in Time Series

Autocorrelation measures correlation between a series and its *lagged* values:
```
ACF(k) = Corr(y_t, y_{t-k})
```

- Slow decay in ACF → strong trend (non-stationary)
- Spikes at seasonal lags (12, 24, ...) → seasonality
- Quickly decays to near zero → stationary

**PACF** (Partial ACF) removes the influence of intermediate lags — helps identify AR order in ARIMA.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF/PACF for raw series (non-stationary)
plot_acf(passengers['Passengers'], lags=36, ax=axes[0,0], color='#1e5fa8')
axes[0,0].set_title('ACF — Raw Passengers\n(slow decay = trend present = non-stationary)')

plot_pacf(passengers['Passengers'], lags=36, ax=axes[0,1], color='#1e5fa8', method='ywm')
axes[0,1].set_title('PACF — Raw Passengers')

# After differencing (remove trend)
diff_pass = np.log(passengers['Passengers']).diff(12).dropna()  # seasonal differencing
plot_acf(diff_pass, lags=36, ax=axes[1,0], color='#1a7a45')
axes[1,0].set_title('ACF — After Seasonal Differencing\n(much more stationary)')

plot_pacf(diff_pass, lags=36, ax=axes[1,1], color='#1a7a45', method='ywm')
axes[1,1].set_title('PACF — After Seasonal Differencing')

plt.suptitle('ACF and PACF: Raw vs Differenced Series', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 Blue shaded region = 95% confidence band for white noise")
print("   Spikes outside the band are statistically significant autocorrelations")

In [ ]:
# Stationarity test — Augmented Dickey-Fuller
print("=== Augmented Dickey-Fuller Test ===")
print("H₀: Series has a unit root (non-stationary)")
print("Reject H₀ if p-value < 0.05\n")

series_dict = {
    'Raw passengers':           passengers['Passengers'],
    'Log passengers':           np.log(passengers['Passengers']),
    'First difference (log)':   np.log(passengers['Passengers']).diff().dropna(),
    'Seasonal diff (log)':      np.log(passengers['Passengers']).diff(12).dropna(),
}

for name, series in series_dict.items():
    adf_result = adfuller(series)
    stationary = adf_result[1] < 0.05
    print(f"{name:<30}  ADF={adf_result[0]:>7.3f}  p={adf_result[1]:.4f}  {'✓ Stationary' if stationary else '✗ Non-stationary'}")
print("\n📌 Seasonal differencing makes the log series stationary")
print("   Stationary series required for ARIMA modeling (next notebook)")

In [ ]:
answers={"q1":"","q2":"","q3":"","q4":"","q5":""}
# q1: Name the four components of a time series
# q2: When would you use multiplicative vs additive decomposition?
# q3: What does a slowly decaying ACF indicate?
# q4: What is the difference between seasonality and a cycle?
# q5: What does 'stationary' mean and why does ARIMA require it?
missing=[k for k,v in answers.items() if not v.strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Time Series Foundations"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
